In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 130)
sns.set_theme(style='whitegrid')

path = r"f:\Christ\Sem 4\AIML\CIA1\Teen_Mental_Health_Dataset_all_column (1).xlsx"
df = pd.read_excel(path)


In [ ]:
# Q1: Load dataset and display first 10 rows
print('Q1: First 10 rows')
display(df.head(10))


In [ ]:
# Q1: Separate numerical vs categorical variables
numeric_cols = df.select_dtypes(include='number').columns.tolist()
categorical_cols = [c for c in df.columns if c not in numeric_cols]
print('Numeric variables:', numeric_cols)
print('Categorical variables:', categorical_cols)


In [ ]:
# Q1: Check dtypes and interpret whether conversion is needed
print('Q1: Data types of all columns')
print(df.dtypes.astype(str))

print('Type-conversion interpretation notes:')
print('- depression_label may be numeric (0/1) stored as float due to missing values -> treat as categorical target for plots/boxplots.')
print('- age may appear float because of missing entries -> convert to int after imputation/rounding if appropriate.')


In [ ]:
# Q2(a): Descriptive stats for numerical variables
print('Q2(a): Descriptive statistics for numerical features')
num_df = df[numeric_cols].copy()
desc = num_df.describe().T[['mean','std','min','25%','50%','75%','max']].round(3)
display(desc)
highest_var_col = desc['std'].idxmax()
print('Highest variation variable (max std):', highest_var_col, 'std=', float(desc.loc[highest_var_col, 'std']))


In [ ]:
# Q2(b): Compare mean and median for daily_social_media_hours and sleep_hours
print('Q2(b): Mean vs Median and skewness')
for col in ['daily_social_media_hours','sleep_hours']:
    if col not in df.columns:
        continue
    mean_val = df[col].mean()
    median_val = df[col].median()
    skew_val = df[col].skew()
    diff = mean_val - median_val
    print('
Column:', col)
    print('Mean  :', round(float(mean_val), 3))
    print('Median:', round(float(median_val), 3))
    print('Skew  :', round(float(skew_val), 3))
    if abs(diff) < 1e-9:
        print('Interpretation: mean≈median -> distribution closer to symmetric.')
    elif diff > 0:
        print('Interpretation: mean>median -> right-skew/positive skew tendency.')
    else:
        print('Interpretation: mean<median -> left-skew/negative skew tendency.')


In [ ]:
# Q3(a): Variables with missing values + count and percentage
print('Q3(a): Missing values by column')
missing = df.isna().sum().to_frame('missing_count')
missing['missing_percent'] = (missing['missing_count'] / len(df) * 100).round(2)
display(missing[missing['missing_count'] > 0].sort_values('missing_count', ascending=False))


In [ ]:
# Q3(b): Imputation for daily_social_media_hours, sleep_hours, screen_time_before_sleep
print('Q3(b): Imputation suggestions for usage/sleep time variables')
print('- Median imputation (robust baseline) for each variable independently.')
print('- KNN/Regression imputation to keep relationships with age/stress/anxiety.')


In [ ]:
# Q3(c): Imputation for stress_level, anxiety_level, addiction_level
print('Q3(c): Imputation suggestions for mental-health score variables')
print('- Median (or mode) imputation because these are typically ordinal/bounded scores.')
print('- Model-based predictive imputation if higher accuracy is required.')


In [ ]:
# Q3(d): MAR vs MNAR explanation
print('Q3(d): MAR vs MNAR for anxiety_level missingness')
print('- MAR: missing anxiety_level depends on observed variables (e.g., low sleep or high screen time) but not on the unseen anxiety value itself.')
print('- MNAR: missing anxiety_level depends on the unobserved anxiety value (e.g., very anxious teens skip the question).')


In [ ]:
# Q4(a): distribution of gender, platform usage, depression label
print('Q4(a): Distributions')
gender_col = 'gender' if 'gender' in df.columns else None
platform_col = 'platform_usage' if 'platform_usage' in df.columns else None
depr_col = 'depression_label' if 'depression_label' in df.columns else None
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (c, title) in zip(axes, [(gender_col,'Gender'), (platform_col,'Platform Usage'), (depr_col,'Depression Label')]):
    if c is None:
        ax.axis('off')
        ax.set_title(title + ' (missing)')
        continue
    sns.countplot(x=c, ax=ax)
    ax.set_title(title)
plt.tight_layout()
plt.show()


In [ ]:
# Q4(b): boxplots for stress_level, anxiety_level, addiction_level grouped by depression_label
print('Q4(b): Boxplots by depression_label')
group_col = 'depression_label'
stress_col = 'stress_level' if 'stress_level' in df.columns else None
anxiety_col = 'anxiety_level' if 'anxiety_level' in df.columns else None
addiction_col = 'addiction_level' if 'addiction_level' in df.columns else None
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (c, title) in zip(axes, [(stress_col,'Stress Level'), (anxiety_col,'Anxiety Level'), (addiction_col,'Addiction Level')]):
    if c is None:
        ax.axis('off')
        ax.set_title(title + ' (missing)')
        continue
    sns.boxplot(x=group_col, y=c, ax=ax)
    ax.set_title(title)
plt.tight_layout()
plt.show()


In [ ]:
# Q4(c): scatter plots
print('Q4(c): Scatter plots')
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
if 'daily_social_media_hours' in df.columns and 'stress_level' in df.columns:
    sns.scatterplot(data=df, x='daily_social_media_hours', y='stress_level',
                    ax=axes[0], hue='depression_label' if 'depression_label' in df.columns else None, legend=False)
    axes[0].set_title('Social Media Hours vs Stress')
if 'sleep_hours' in df.columns and 'anxiety_level' in df.columns:
    sns.scatterplot(data=df, x='sleep_hours', y='anxiety_level',
                    ax=axes[1], hue='depression_label' if 'depression_label' in df.columns else None, legend=False)
    axes[1].set_title('Sleep Hours vs Anxiety')
plt.tight_layout()
plt.show()


In [ ]:
# Q4(d): interpret any three charts
print('Q4(d): Interpretation')
print('1) Gender/platform/depression distributions show which group dominates the dataset.')
print('2) If depression_label=1 has higher medians in boxplots for stress/anxiety/addiction, it suggests worse mental health for depressed teens.')
print('3) Scatter plots help reveal direction: higher social media hours may align with higher stress; higher sleep may align with lower anxiety.')


In [ ]:
# Q5: correlation analysis on numerical variables
print('Q5: Correlation matrix (numerical features)')
corr = df[numeric_cols].corr(numeric_only=True)
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

target_list = ['daily_social_media_hours','sleep_hours','stress_level','anxiety_level','addiction_level','academic_performance']
target_list = [c for c in target_list if c in df.columns]
if 'daily_social_media_hours' in corr.columns:
    print('Correlations with daily_social_media_hours')
    display(corr.loc['daily_social_media_hours', target_list].sort_values(ascending=False).round(3))

print('Categorical variables should not be directly included in correlation because correlation assumes numeric meaning; categorical codes can create misleading linear relationships unless encoded.')


In [ ]:
# Q6(a): outliers by IQR for required variables
print('Q6(a): Outliers by IQR')
cols = ['daily_social_media_hours','sleep_hours','stress_level','anxiety_level']
for c in cols:
    if c not in df.columns:
        continue
    q1 = df[c].quantile(0.25)
    q3 = df[c].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    out_count = df[(df[c] < lower) | (df[c] > upper)][c].dropna().shape[0]
    print(c, ': lower=', round(float(lower),3), 'upper=', round(float(upper),3), 'outliers_count=', out_count)


In [ ]:
# Q6(b): extreme values errors or behavior
print('Q6(b): Are extreme values errors or meaningful behavior?')
print('- Check if values are outside realistic ranges (possible data entry errors).')
print('- If values are plausible, they may represent real behavioral patterns (e.g., very high screen time or very low sleep).')


In [ ]:
# Q7(a-d): final analysis answers
print('Q7(a): Social media usage vs stress/anxiety/addiction')
print('- Based on correlation/scatter, positive association suggests higher social media use aligns with higher stress/anxiety/addiction.')

print('Q7(b): Sleep habits & screen time before sleep')
print('- Shorter sleep and more screen time before sleep generally associate with worse mental health indicators.')

print('Q7(c): Factors contributing to depression')
print('- Stress_level, anxiety_level, and addiction_level are usually strongest contributors; also low sleep/high usage can contribute indirectly.')

print('Q7(d): Compare academic performance/physical/social across depression categories')
feature_candidates = ['academic_performance','physical_activity_level','social_interaction_level','physical_activity','social_interaction']
feature_candidates = [c for c in feature_candidates if c in df.columns]
if 'depression_label' in df.columns and len(feature_candidates) > 0:
    display(df.groupby('depression_label')[feature_candidates].median(numeric_only=True).round(3))
else:
    print('Required columns for group comparison are not found in this dataset by expected names.')
